# 🧪 Module 1.3: Your First MLFlow Experiment

**Goal**: Train a simple ML model, log everything to MLFlow, and view results in the UI.

This is the "Hello World" of MLFlow — by the end of this notebook, you'll understand the basic workflow you'll use throughout this entire course.

---

## The MLFlow Workflow (3 Steps)

Every MLFlow experiment follows this pattern:

```
1. SET UP    →  Create/select an experiment
2. LOG       →  Log parameters, metrics, artifacts, and models
3. REVIEW    →  View results in the MLFlow UI
```

Let's do each step!

## Step 1: Import Libraries

In [1]:
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

import pandas as pd
import numpy as np

print(f"MLFlow version: {mlflow.__version__}")
print("✅ All imports successful!")

MLFlow version: 3.14.0
✅ All imports successful!


In [2]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: sqlite:///c:/Users/sujat/projects/MLFlow_Learn/mlflow.db


## Step 2: Load and Prepare Data

We'll use the classic **Iris dataset** — a simple classification problem with 3 flower species.

In [3]:
# Load the Iris dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target classes: {list(iris.target_names)}")
print(f"\nFirst 5 rows:")
X.head()

Dataset shape: (150, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

First 5 rows:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [4]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 120
Test samples: 30


## Step 3: Create an MLFlow Experiment

An **experiment** is a named container for related runs. Think of it as a project folder.

> 💡 If the experiment doesn't exist, `set_experiment()` creates it. If it exists, it selects it.

In [5]:
# Create (or select) an experiment
mlflow.set_experiment("01_first_experiment")

print("✅ Experiment '01_first_experiment' is active!")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/06/23 10:50:38 INFO mlflow.tracking.fluent: Experiment with name '01_first_experiment' does not exist. Creating a new experiment.


✅ Experiment '01_first_experiment' is active!
Tracking URI: sqlite:///c:/Users/sujat/projects/MLFlow_Learn/mlflow.db


## Step 4: Train a Model and Log to MLFlow

Here's the core pattern. We use `mlflow.start_run()` as a context manager to:
1. **Start a run** — creates a new tracking record
2. **Log parameters** — the inputs/configuration we chose
3. **Train the model** — regular scikit-learn code
4. **Log metrics** — how well the model performed
5. **Log the model** — save the trained model as an artifact
6. **End the run** — automatically done when the `with` block exits

In [6]:
# Define our hyperparameters
params = {
    "max_depth": 3,
    "min_samples_split": 5,
    "random_state": 42
}

# Start an MLFlow run
with mlflow.start_run(run_name="first_decision_tree"):
    
    # 1. Log parameters (the inputs we chose)
    mlflow.log_params(params)
    
    # 2. Train the model (regular sklearn code)
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    
    # 3. Make predictions
    y_pred = model.predict(X_test)
    
    # 4. Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # 5. Log metrics (how well the model performed)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # 6. Log the trained model as an artifact
    mlflow.sklearn.log_model(model, artifact_path="model")
    
    # 7. Add a tag for context
    mlflow.set_tag("model_type", "DecisionTree")
    mlflow.set_tag("dataset", "iris")
    
    # Print results
    run_id = mlflow.active_run().info.run_id
    print(f"🎉 Run completed!")
    print(f"   Run ID: {run_id}")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")

2026/06/23 11:53:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🎉 Run completed!
   Run ID: 52817c6a880143b6a498556bc7037480
   Accuracy: 1.0000
   F1 Score: 1.0000


## Step 5: Try Different Hyperparameters

Let's run a few more experiments with different settings to see how MLFlow tracks multiple runs.

In [7]:
# Try multiple configurations
configurations = [
    {"max_depth": 2, "min_samples_split": 2, "random_state": 42},
    {"max_depth": 5, "min_samples_split": 3, "random_state": 42},
    {"max_depth": 10, "min_samples_split": 10, "random_state": 42},
    {"max_depth": None, "min_samples_split": 2, "random_state": 42},  # No depth limit
]

print("Running multiple experiments...\n")

for i, params in enumerate(configurations, 1):
    with mlflow.start_run(run_name=f"decision_tree_v{i}"):
        
        # Log parameters
        # Note: log_params requires all values to be strings, but MLFlow
        # handles conversion automatically for common types
        mlflow.log_params(params)
        
        # Train
        model = DecisionTreeClassifier(**params)
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        
        # Log model
        mlflow.sklearn.log_model(model, artifact_path="model")
        
        # Tags
        mlflow.set_tag("model_type", "DecisionTree")
        mlflow.set_tag("dataset", "iris")
        
        print(f"  Run {i}: max_depth={str(params['max_depth']):>4}, "
              f"accuracy={accuracy:.4f}, f1={f1:.4f}")

print("\n✅ All runs completed! Check the MLFlow UI to compare them.")

2026/06/23 11:55:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Running multiple experiments...



2026/06/23 11:56:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Run 1: max_depth=   2, accuracy=0.9667, f1=0.9664
  Run 2: max_depth=   5, accuracy=1.0000, f1=1.0000


2026/06/23 11:56:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/23 11:56:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Run 3: max_depth=  10, accuracy=1.0000, f1=1.0000
  Run 4: max_depth=None, accuracy=1.0000, f1=1.0000

✅ All runs completed! Check the MLFlow UI to compare them.


## Step 6: View Results Programmatically

You can also query your runs from code using `mlflow.search_runs()`.

In [8]:
# Search all runs in our experiment
runs = mlflow.search_runs(
    experiment_names=["01_first_experiment"],
    order_by=["metrics.accuracy DESC"]
)

# Display key columns
display_cols = [
    "run_id", 
    "params.max_depth", 
    "params.min_samples_split",
    "metrics.accuracy", 
    "metrics.f1_score",
    "tags.mlflow.runName"
]

# Filter to only existing columns
display_cols = [c for c in display_cols if c in runs.columns]

print("📊 All Runs (sorted by accuracy):")
runs[display_cols]

📊 All Runs (sorted by accuracy):


,run_id,params.max_depth,params.min_samples_split,metrics.accuracy,metrics.f1_score,tags.mlflow.runName
0,00791f1ec73244b691de7ced473ba148,None,2,1.000000,1.000000,decision_tree_v4
1,5ec14d2fe76b4216b02add1e55e0ad43,10,10,1.000000,1.000000,decision_tree_v3
2,8a29e9934fad442a904cfc92bc5496e8,5,3,1.000000,1.000000,decision_tree_v2
3,52817c6a880143b6a498556bc7037480,3,5,1.000000,1.000000,first_decision_tree
4,2be72a70fc98423f960ba39ac83094ff,2,2,0.966667,0.966411,decision_tree_v1


## Step 7: View in the MLFlow UI 🖥️

Now let's see our results visually!

### Launch the UI

Open a **new terminal** and run:

```bash
cd c:\Users\sujat\projects\MLFlow_Learn
mlflow ui
```

Then open your browser to: **http://localhost:5000**

### What to Look For

1. **Experiments sidebar** (left) → Click on `01_first_experiment`
2. **Runs table** → See all 5 runs with their parameters and metrics
3. **Compare runs** → Select multiple runs and click "Compare"
4. **Charts** → Click on a metric to see a chart view
5. **Run details** → Click on any run to see its parameters, metrics, artifacts, and logged model

## Step 8: Load a Logged Model

One of the best features of MLFlow — you can reload any model you've logged!

In [9]:
# Find the best run
best_run = runs.iloc[0]  # Already sorted by accuracy DESC
best_run_id = best_run["run_id"]

print(f"Best run ID: {best_run_id}")
print(f"Best accuracy: {best_run['metrics.accuracy']:.4f}")

# Load the model from MLFlow
loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

# Make predictions with the loaded model
new_predictions = loaded_model.predict(X_test[:5])

print(f"\nPredictions from loaded model: {new_predictions}")
print(f"Actual values:                 {y_test[:5]}")
print("\n✅ Model loaded and working!")

Best run ID: 00791f1ec73244b691de7ced473ba148
Best accuracy: 1.0000



Predictions from loaded model: [1 0 2 1 1]
Actual values:                 [1 0 2 1 1]

✅ Model loaded and working!


## 🔑 Key Takeaways

1. **`mlflow.set_experiment()`** — creates or selects an experiment
2. **`mlflow.start_run()`** — starts tracking a new run (use as context manager with `with`)
3. **`mlflow.log_params()`** — logs input parameters (hyperparameters, config)
4. **`mlflow.log_metric()`** — logs output metrics (accuracy, loss, etc.)
5. **`mlflow.sklearn.log_model()`** — saves the trained model as an artifact
6. **`mlflow.search_runs()`** — queries past runs programmatically
7. **`mlflow.sklearn.load_model()`** — loads a previously saved model

### The Pattern You'll Use Every Time

```python
mlflow.set_experiment("my_experiment")

with mlflow.start_run():
    mlflow.log_params({...})       # What you configured
    model.fit(X_train, y_train)     # Train (your regular code)
    mlflow.log_metric("acc", acc)   # How it performed
    mlflow.sklearn.log_model(...)   # Save the model
```

---

## 🎓 Module 1 Complete!

You've learned:
- What MLFlow is and why it matters
- How to set up and verify your environment
- How to track experiments with the basic MLFlow API
- How to view results in the UI and reload models

**Next up**: Module 2 — Deep dive into the **Tracking API** →